# Packages

In [8]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from constants import Constant as C
from loaders import load_ratings
from models import ContentBased
from python_helper import submit_predictions
# `submit_predictions` auto-loads .env from the repo root on import,
# so HACKATHON_URL and HACKATHON_TOKEN become available without any
# manual os.environ setup.

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# How to submit predictions to the hackathon

## One-time setup: create a `.env` file at the repo root

Add the URL and your group's token (provided by the instructor):

```env
HACKATHON_URL=https://recsys-hackathon.vercel.app
HACKATHON_TOKEN=your-group-token-here
```

`.env` is gitignored. If you change it, restart the Jupyter kernel to pick up the new values (the helper loads `.env` once at import time).

## Workflow

1. **Tune offline.** Use your evaluator notebook to pick the best `feature_method` and `regressor_method` for `ContentBased`. Every submission to the server counts toward your quota — iterate locally before submitting.
2. **Predict.** Run `make_hackathon_prediction(...)` below to produce `df_predictions`.
3. **Submit.** Run `submit_predictions(df_predictions)` — the server scores it and prints your RMSE, rank, and remaining quota.

## Quota per group

`10s` cooldown between submissions · `10` per hour · `300` total over the hackathon. **Errors count toward the quota**, so verify the CSV format works once before iterating in a tight loop.

In [ ]:
def make_hackathon_prediction(feature_method, regressor_method):
    """Train a ContentBased model and produce predictions on the test set.

    Returns a DataFrame with columns ['userId', 'movieId', 'rating'] in the
    exact order of the hidden test set. Pass it to submit_predictions().
    """
    # 1) load train data - make sure to redirect the DATA_PATH to 'data/hackathon'
    assert str(C.DATA_PATH) == 'data/hackathon'
    sp_ratings = load_ratings(surprise_format=True)
    train_set = sp_ratings.build_full_trainset()

    # 2) train your ContentBased model on the train set
    content_knn = ContentBased(feature_method, regressor_method)
    content_knn.fit(train_set)

    # 3) make predictions on the test set
    df_test = pd.read_csv('data/hackathon/evidence/ratings_test.csv')[C.USER_ITEM_RATINGS]
    test_records = list(df_test.to_records(index=False))
    predictions = content_knn.test(test_records)
    output_predictions = []
    for uid, iid, _, est, _ in predictions:
        output_predictions.append([uid, iid, est])
    df_predictions = pd.DataFrame(data=output_predictions, columns=df_test.columns)

    return df_predictions


#### Predict and submit (for hackathon) ####
#### NOTE : these predictions were meant to be run one at a time during the hackathon. 
#### The retained predictions are located in the evaluator

#df_predictions = make_hackathon_prediction('title_length', 'linear_regression_true')
#df_predictions = make_hackathon_prediction('genome_scaled_tags', 'ridge_cv_centered')
#df_predictions = make_hackathon_prediction('all_content_full', 'ridge_cv')
#df_predictions = make_hackathon_prediction('all_content_tmdb', 'ridge_cv')
#df_predictions = make_hackathon_prediction('all_content_tmdb', 'ridge_cv')## 
#df_predictions = make_hackathon_prediction('all_content_tmdb_tags1128', 'ridge_cv')

#### Best current prediction ####
df_predictions = make_hackathon_prediction('all_content_tmdb_tags2000', 'ridge_cv')
df_predictions.head()

,userId,movieId,rating
0,277,1,4.302017
1,277,41,4.534920
2,277,47,4.406038
3,277,76,3.776864
4,277,141,3.768088


In [ ]:
submit_predictions(df_predictions)

------------------------------------------------------------
  [OK] Submission accepted
       RMSE this submission   0.7279
       Best RMSE for group    0.7279
       Current rank           #5
       Remaining this hour    9/10
       Remaining total        289/300
------------------------------------------------------------


{'status': 'ok',
 'rmse': 0.7279210745504656,
 'error': None,
 'rank': 5,
 'best_rmse': 0.7278928341297375,
 'n_rows': 104468,
 'remaining': {'hour': 9, 'total': 289}}

In [ ]:
# Optional: peek at your remaining quota without burning a submission
from python_helper import check_quota
check_quota()

------------------------------------------------------------
  [QUOTA] Group 3
       This hour          1/10 used  (9 remaining)
       Total              9/300 used  (291 remaining)
       Last submission    2026-05-12T13:27:30.079Z
       Cooldown           clear — you can submit now
------------------------------------------------------------


{'group': {'id': 8, 'name': 'Group 3'},
 'hour_used': 1,
 'hour_remaining': 9,
 'hour_limit': 10,
 'total_used': 9,
 'total_remaining': 291,
 'total_limit': 300,
 'cooldown_seconds': 10,
 'last_submitted_at': '2026-05-12T13:27:30.079Z',
 'next_allowed_at': None}